# MLB win-probability model

The project's **sport-agnostic** Elo+feature pipeline (`walk_forward`) on **38k real MLB games** (2010–2026, `data/mlb.duckdb`, free MLB Stats API), **plus a starting-pitcher feature** — the dominant MLB lever team-Elo can't see. Out-of-sample throughout (fit earlier games, score later).

In [ ]:
import sys
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import os as _os
if _os.getenv('NB_DARK'):
    sns.set_theme(style='darkgrid', rc={
        'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','savefig.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','text.color':'#c9d1d9',
        'axes.titlecolor':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e',
        'grid.color':'#21262d'})
else:
    sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'src'/'sportsball').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src')); sys.path.insert(0, str(ROOT/'scripts')); sys.path.insert(0, str(ROOT/'research'/'mlb'))
from sportsball.pipelines._elo import walk_forward
from train_eval_duckdb import _pipeline
from pitcher_features import pitcher_run_prevention
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score
K, HFA = 4, 24

In [ ]:
con = duckdb.connect(str(ROOT/'data'/'mlb.duckdb'), read_only=True)
rows = con.execute('''SELECT game_date,home_team,away_team,home_score,away_score,
    home_sp_id,away_sp_id,home_sp,away_sp FROM games ORDER BY game_date, game_pk''').fetchall()
con.close()
results = [(r[0],r[1],r[2],r[3],r[4]) for r in rows]                      # for Elo walk
pgames  = [(r[5],r[6],r[3],r[4]) for r in rows]                          # (h_sp,a_sp,hs,as)
frows, snaps = walk_forward(results, K, HFA, mov_enabled=True, carry=0.75, gap_days=120, form_window=15)
X = np.array([r.features for r in frows]); y = np.array([1 if r.actual>=1 else 0 for r in frows])
elo_p = np.array([r.exp_home for r in frows])
pitch = np.array(pitcher_run_prevention(pgames, window=10)['pitcher_adv_home'])  # aligned row-for-row
Xp = np.column_stack([X, pitch])                                          # 9 features + pitcher
cut = int(0.85*len(X))
print(f'{len(X)} games | test {len(y)-cut} | home-win base rate {y[cut:].mean():.3f}')

## Does the starting pitcher help? (out-of-sample holdout)

Reference points stacked up: always-home, raw Elo, the 9-feature logistic, the same **+ pitcher**, and pitcher-advantage **alone**.

In [ ]:
def rep(name, p):
    return {'model':name,'accuracy':round(accuracy_score(y[cut:],(p[cut:]>=0.5).astype(int)),4),
            'log_loss':round(log_loss(y[cut:],p[cut:],labels=[0,1]),4),'brier':round(brier_score_loss(y[cut:],p[cut:]),4)}
def fit_p(M):
    m=_pipeline().fit(M[:cut], y[:cut]); out=np.empty(len(y)); out[:cut]=m.predict_proba(M[:cut])[:,1]; out[cut:]=m.predict_proba(M[cut:])[:,1]; return out
home_const = np.full(len(y), y[:cut].mean())
full_p   = fit_p(X)
pitch_p  = fit_p(Xp)
ponly_p  = fit_p(pitch.reshape(-1,1))
tbl = pd.DataFrame([rep('always-home (base)', home_const), rep('Elo only', elo_p),
                    rep('9-feature logistic', full_p), rep('9-feat + pitcher', pitch_p),
                    rep('pitcher advantage only', ponly_p)])
tbl

In [ ]:
d_acc = tbl.loc[3,'accuracy']-tbl.loc[2,'accuracy']; d_ll = tbl.loc[3,'log_loss']-tbl.loc[2,'log_loss']
print(f'pitcher feature lift over 9-feature: accuracy {d_acc:+.4f}, log_loss {d_ll:+.4f}')
verdict = ('meaningfully helps' if d_ll < -0.002 else
           'marginal / within-noise (the runs-allowed proxy adds little beyond team-Elo)' if d_ll < 0 else
           'no out-of-sample gain')
print('->', verdict)

## Calibration (9-feat + pitcher)

In [ ]:
from sklearn.calibration import calibration_curve
fp, mp = calibration_curve(y[cut:], pitch_p[cut:], n_bins=8, strategy='quantile')
plt.figure(figsize=(6,6)); plt.plot([0,1],[0,1],'--',color='gray',label='perfect')
plt.plot(mp, fp, 'o-', label='MLB model (+pitcher)'); plt.xlabel('predicted P(home win)'); plt.ylabel('observed')
plt.title('Calibration (holdout)'); plt.legend(); plt.show()

## Sanity: best starters by run prevention

Career run-prevention in this dataset (runs the opponent scored in their starts; lower = better, ≥80 starts). Should surface recognizable aces — a check that the feature tracks reality.

In [ ]:
ra = {}
for r in rows:
    if r[7]: ra.setdefault(r[7], []).append(r[4])   # home_sp name -> away runs
    if r[8]: ra.setdefault(r[8], []).append(r[3])   # away_sp name -> home runs
agg = pd.DataFrame([(n, np.mean(v), len(v)) for n,v in ra.items()], columns=['pitcher','runs_allowed','starts'])
best = agg[agg.starts>=80].sort_values('runs_allowed').head(12)
plt.figure(figsize=(9,6)); sns.barplot(data=best, y='pitcher', x='runs_allowed', color='seagreen')
plt.xlabel('mean opponent runs / start (lower = better)'); plt.title('Best run-prevention starters (>=80 starts)'); plt.tight_layout(); plt.show()

## Team strength (final Elo) & skill over time

In [ ]:
rate = pd.DataFrame([(t, s.elo) for t,s in snaps.items()], columns=['team','elo']).sort_values('elo', ascending=False)
fig, ax = plt.subplots(1,2, figsize=(13,5))
sns.barplot(data=rate.head(8), y='team', x='elo', ax=ax[0], color='seagreen'); ax[0].set_title('Top 8 Elo')
sns.barplot(data=rate.tail(8), y='team', x='elo', ax=ax[1], color='indianred'); ax[1].set_title('Bottom 8 Elo')
for a in ax: a.set_xlim(rate.elo.min()-20, rate.elo.max()+20)
plt.tight_layout(); plt.show()
win=400; correct=(pitch_p[cut:]>=0.5).astype(int)==y[cut:]
plt.figure(figsize=(11,4)); plt.plot(pd.Series(correct).rolling(win,min_periods=50).mean().values, color='seagreen', label='model (+pitcher)')
plt.plot(pd.Series(y[cut:]==1).rolling(win,min_periods=50).mean().values, color='gray', ls='--', label='always-home')
plt.axhline(0.5,color='crimson',lw=.6); plt.legend(); plt.ylabel(f'rolling accuracy ({win})'); plt.title('Skill over time'); plt.show()

## Verdict

Team-Elo gives real-but-modest skill (~56%, capped because it's blind to the starting pitcher). The run-prevention **pitcher proxy** has *some* signal (pitcher-advantage alone beats always-home), but as an add-on it's **marginal / within noise** — see the holdout lift above. The reason: 'opponent runs when this starter pitched' folds in the bullpen, defense, and park, so it barely isolates the pitcher beyond what team-Elo already encodes.

**The real unlock is a true pitcher rating** — FIP / K-BB% / xERA from per-start pitcher game logs (MLB Stats API `people/{id}/stats?stats=gameLog&group=pitching`), which isolates the pitcher from his team. That's a heavier fetch (~per-pitcher, per-season) but is the honest path to moving the needle. Ingested closing odds (`market_logit`) would also help. Re-run `research/mlb/ingest_mlb.py` to refresh the live 2026 season.